In [13]:
from google.colab import drive
import os

drive.mount('/content/drive')

caminho = '/content/drive/MyDrive/Aula_llm/Trabalho final/Dados/IBM'

caminho_accounts = '/content/drive/MyDrive/Aula_llm/Trabalho final/Dados/IBM/LI-Small_accounts.csv'
caminho_trans = '/content/drive/MyDrive/Aula_llm/Trabalho final/Dados/IBM/LI-Small_Trans.csv'

print(os.path.exists(caminho_accounts))
print(os.path.exists(caminho_trans))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
True
True


In [15]:
len(df_trans), len(df_accounts)

(6924049, 712688)

In [16]:
sum(df_trans['Is Laundering'] == 1)

3565

In [12]:
import pandas as pd

df_trans = pd.read_csv(caminho_trans)

In [ ]:
colunms_trans = df_trans.columns

In [ ]:
colunms_trans

Index(['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1',
       'Amount Received', 'Receiving Currency', 'Amount Paid',
       'Payment Currency', 'Payment Format', 'Is Laundering'],
      dtype='object')

In [ ]:
df_resultado = df_trans.copy()
df_resultado.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
3,2022/09/01 00:16,3814,8006AD080,3814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,0
4,2022/09/01 00:00,20,8006AD530,20,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,0


In [ ]:
print(df_resultado.shape)

memoria_gb = df_resultado.memory_usage(deep=True).sum() / 1024**3
print(f"Memória usada: {memoria_gb:.2f} GB")

print(df_resultado["Is Laundering"].value_counts())

(6924049, 11)
Memória usada: 2.52 GB
Is Laundering
0    6920484
1       3565
Name: count, dtype: int64


In [ ]:
df_resultado.columns

Index(['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1',
       'Amount Received', 'Receiving Currency', 'Amount Paid',
       'Payment Currency', 'Payment Format', 'Is Laundering'],
      dtype='object')

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

TARGET = "Is Laundering"


def criar_variaveis_ibm_para_grafo(df):
    df = df.copy()
    df.columns = df.columns.str.strip()

    # =========================
    # 1. Data
    # =========================
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

    ts = df["Timestamp"]

    df["ano"] = ts.dt.year
    df["mes"] = ts.dt.month
    df["dia_mes"] = ts.dt.day
    df["hora"] = ts.dt.hour
    df["minuto"] = ts.dt.minute
    df["dia_semana"] = ts.dt.dayofweek
    df["semana_ano"] = ts.dt.isocalendar().week.astype("float64")
    df["semana_mes"] = ((ts.dt.day - 1) // 7 + 1).astype("float64")

    df["fim_de_semana"] = df["dia_semana"].isin([5, 6]).astype(int)

    df["tipo_semana"] = np.where(
        df["fim_de_semana"] == 1,
        "fim_de_semana",
        "meio_de_semana"
    )

    h = df["hora"]

    df["periodo_dia"] = np.select(
        [
            h.between(0, 5),
            h.between(6, 11),
            h.between(12, 17),
            h.between(18, 23)
        ],
        [
            "madrugada",
            "manha",
            "tarde",
            "noite"
        ],
        default="desconhecido"
    )

    df["periodo_dia_3"] = np.select(
        [
            h.between(6, 11),
            h.between(12, 17)
        ],
        [
            "manha",
            "tarde"
        ],
        default="noite"
    )

    df["inicio_mes"] = ts.dt.is_month_start.astype("float64")
    df["fim_mes"] = ts.dt.is_month_end.astype("float64")

    # Representação cíclica
    df["hora_sin"] = np.sin(2 * np.pi * df["hora"] / 24)
    df["hora_cos"] = np.cos(2 * np.pi * df["hora"] / 24)

    df["dia_semana_sin"] = np.sin(2 * np.pi * df["dia_semana"] / 7)
    df["dia_semana_cos"] = np.cos(2 * np.pi * df["dia_semana"] / 7)

    df["mes_sin"] = np.sin(2 * np.pi * df["mes"] / 12)
    df["mes_cos"] = np.cos(2 * np.pi * df["mes"] / 12)

    # =========================
    # 2. Valores monetários
    # =========================
    df["Amount Received"] = pd.to_numeric(df["Amount Received"], errors="coerce")
    df["Amount Paid"] = pd.to_numeric(df["Amount Paid"], errors="coerce")

    df["dif_valor"] = df["Amount Paid"] - df["Amount Received"]
    df["dif_valor_abs"] = df["dif_valor"].abs()

    df["razao_pago_recebido"] = (
        df["Amount Paid"] / (df["Amount Received"] + 1e-6)
    )

    df["amount_paid_log"] = np.log1p(df["Amount Paid"].clip(lower=0))
    df["amount_received_log"] = np.log1p(df["Amount Received"].clip(lower=0))

    # =========================
    # 3. Relações origem-destino
    # =========================
    df["mesmo_banco"] = (
        df["From Bank"].astype(str) == df["To Bank"].astype(str)
    ).astype(int)

    df["mesma_conta"] = (
        df["Account"].astype(str) == df["Account.1"].astype(str)
    ).astype(int)

    df["mesma_moeda"] = (
        df["Payment Currency"].astype(str) == df["Receiving Currency"].astype(str)
    ).astype(int)

    df["pagamento_reinvestment"] = (
        df["Payment Format"].astype(str).str.lower() == "reinvestment"
    ).astype(int)

    # =========================
    # 4. IDs para grafo
    # =========================
    df["source_account_id"] = (
        df["From Bank"].astype(str).str.strip()
        + "_"
        + df["Account"].astype(str).str.strip()
    )

    df["target_account_id"] = (
        df["To Bank"].astype(str).str.strip()
        + "_"
        + df["Account.1"].astype(str).str.strip()
    )

    # ID único da transação
    df["transaction_id"] = np.arange(len(df))

    return df

In [ ]:
import gc
TARGET = "Is Laundering"

df_resultado[TARGET] = df_resultado[TARGET].astype("int8")

df_fraudes = df_resultado[df_resultado[TARGET] == 1]
df_normais = df_resultado[df_resultado[TARGET] == 0]

print("Fraudes:", len(df_fraudes))
print("Normais:", len(df_normais))

proporcao_normais = 10

qtd_normais = min(
    len(df_normais),
    len(df_fraudes) * proporcao_normais,
    100_000
)

df_normais_amostra = df_normais.sample(
    n=qtd_normais,
    random_state=42
)

df_grafo = pd.concat(
    [df_fraudes, df_normais_amostra],
    axis=0
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(df_grafo.shape)
print(df_grafo[TARGET].value_counts())

del df_fraudes
del df_normais
del df_normais_amostra
gc.collect()

Fraudes: 3565
Normais: 6920484
(39215, 11)
Is Laundering
0    35650
1     3565
Name: count, dtype: int64


27702

In [ ]:
df_resultado['Is Laundering'].value_counts()

,count
Is Laundering,
0,6920484
1,3565


In [ ]:
proporcao_normais = 10

qtd_normais = min(
    len(df_normais),
    len(df_fraudes) * proporcao_normais,
    100_000
)

df_normais_amostra = df_normais.sample(
    n=qtd_normais,
    random_state=42
)

df_grafo = pd.concat([df_fraudes, df_normais_amostra], axis=0)

df_grafo = df_grafo.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(df_grafo.shape)
print(df_grafo[TARGET].value_counts())

(39215, 11)
Is Laundering
0    35650
1     3565
Name: count, dtype: int64


In [ ]:
import os
import pandas as pd
import numpy as np

TARGET = "Is Laundering"

caminho = "/content/drive/MyDrive/Aula_llm/Trabalho final/Dados/IBM/"
os.makedirs(caminho, exist_ok=True)

In [ ]:
df_grafo_tratado = criar_variaveis_ibm_para_grafo(df_grafo)

print(df_grafo_tratado.head())
print(df_grafo_tratado.shape)

print(df_grafo_tratado[[
    "source_account_id",
    "target_account_id",
    "Account",
    "Account.1",
    "Is Laundering"
]].head())

            Timestamp  From Bank    Account  To Bank  Account.1  \
0 2022-09-10 15:48:00      14172  8025B5F50    22693  80A9F7D70   
1 2022-09-02 02:34:00         70  10042B660   237246  8172A5260   
2 2022-09-07 10:46:00        498  800DAA850    21876  801C37770   
3 2022-09-09 10:04:00         70  10042B660    62871  8197FBD50   
4 2022-09-09 14:18:00      25304  8075B8250    24070  80CDA2C00   

   Amount Received Receiving Currency  Amount Paid Payment Currency  \
0             9.56               Euro         9.56             Euro   
1        152800.97          US Dollar    152800.97        US Dollar   
2           144.20               Euro       144.20             Euro   
3          1028.05          US Dollar      1028.05        US Dollar   
4        181930.18               Euro    181930.18             Euro   

  Payment Format  ...  razao_pago_recebido  amount_paid_log  \
0         Cheque  ...                  1.0         2.357073   
1         Cheque  ...                  1.0  

In [ ]:
y = df_grafo_tratado[TARGET].astype(int)

df_train, df_temp = train_test_split(
    df_grafo_tratado,
    test_size=0.30,
    random_state=42,
    stratify=y
)

df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp[TARGET]
)

print("Treino:")
print(df_train[TARGET].value_counts())

print("\nValidação:")
print(df_val[TARGET].value_counts())

print("\nTeste:")
print(df_test[TARGET].value_counts())

Treino:
Is Laundering
0    24955
1     2495
Name: count, dtype: int64

Validação:
Is Laundering
0    5347
1     535
Name: count, dtype: int64

Teste:
Is Laundering
0    5348
1     535
Name: count, dtype: int64


Arquivos para grafo salvos com sucesso!


In [ ]:
import os
import pandas as pd

TARGET = "Is Laundering"

caminho = "/content/drive/MyDrive/Aula_llm/Trabalho final/Dados/IBM/"
os.makedirs(caminho, exist_ok=True)

df_train.to_csv(
    caminho + 'dados_trans_treino.csv',
    index=False
)

df_val.to_csv(
    caminho + 'dados_trans_validacao.csv',
    index=False
)

df_test.to_csv(
    caminho + 'dados_trans_teste.csv',
    index=False
)


print("Arquivos salvos com sucesso!")

Arquivos salvos com sucesso!


# Contas de clientes

In [5]:
import pandas as pd
df_accounts = pd.read_csv(caminho_accounts)
df_accounts

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,China Bank #2820,314693,81B86A280,800D8CCF0,Corporation #41344
1,France Bank #4585,311253,8187FEA80,800B505E0,Corporation #54497
2,China Bank #2242,39996,803961E00,800D03F60,Partnership #36904
3,National Bank of Newport,331440,81B075800,801567C10,Corporation #16224
4,UK Bank #33,135417,80CF87C80,801085E00,Partnership #72930
...,...,...,...,...,...
712683,China Bank #42,692,80346A5F0,800D080A0,Corporation #40025
712684,First Bank of Watertown,118699,80B802A70,8005319C0,Partnership #14803
712685,Bank of Lincoln,213123,80847BE70,800453480,Partnership #11893
712686,Hilltop Credit Union,18747,803A36270,8003B5120,Partnership #6726


In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

df_accounts['Bank Type'] = df_accounts['Bank Name'].str.split('#').str[0].str.strip()
df_accounts['Entity Type'] = df_accounts['Entity Name'].str.split('#').str[0].str.strip()

# 3. Codificando as variáveis categóricas (Texto -> Número)
le_bank_type = LabelEncoder()
le_entity_type = LabelEncoder()
le_entity_id = LabelEncoder()

df_accounts['Bank Type_Encoded'] = le_bank_type.fit_transform(df_accounts['Bank Type'])
df_accounts['Entity Type_Encoded'] = le_entity_type.fit_transform(df_accounts['Entity Type'])
df_accounts['Entity ID_Encoded'] = le_entity_id.fit_transform(df_accounts['Entity ID'])

# 4. Mantendo e Normalizando o Bank ID
scaler = StandardScaler()
# O Bank ID permanece aqui, mas escalonado para a GNN performar melhor
df_accounts['Bank ID_Scaled'] = scaler.fit_transform(df_accounts[['Bank ID']])

# 5. Definindo o ID do Nó (Mapeamento de Account Number para índices 0, 1, 2...)
# GNNs exigem que os nós sejam indexados de 0 a N-1
le_account = LabelEncoder()
df_accounts['node_id'] = le_account.fit_transform(df_accounts['Account Number'])

node_features = df_accounts[['node_id', 'Account Number', 'Bank ID_Scaled', 'Bank Type_Encoded', 'Entity ID_Encoded', 'Entity Type_Encoded']]

print("Tabela Pronta para a GNN:")
print(node_features.to_string(index=False))

A saída de streaming foi truncada nas últimas 5000 linhas.
  604460      817F12290        0.310287                396             210282                    0
  145777      805B9B310       -0.960625                371             209246                    0
  451829      811D27A90       -0.052742                218              85602                    2
  292053      80B7D8190       -0.065274                328             158357                    2
  686229      81B1D6191        1.984563                124              70894                    2
  603714      817E9BA90       -0.472570                396             210106                    3
  303650      80BF10890       -0.729975                174             170247                    0
  376773      80ED78810       -0.788942                266             160997                    0
  505977      813FDA290        0.226931                218             128774                    3
   56468      80239E610       -0.863752           

In [9]:
caminho = "/content/drive/MyDrive/Aula_llm/Trabalho final/Dados/IBM/"
df_accounts.to_csv(
    caminho + 'dados_account_tratada.csv',
    index=False
)